In [ ]:
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json
import ast

In [ ]:
def rpy_to_rotmat(rpy):
    """
    rpy: (..., 3) roll, pitch, yaw
    returns: (..., 3, 3) rotation matrix (base -> world)
    """
    roll, pitch, yaw = rpy[..., 0], rpy[..., 1], rpy[..., 2]

    cr, sr = np.cos(roll),  np.sin(roll)
    cp, sp = np.cos(pitch), np.sin(pitch)
    cy, sy = np.cos(yaw),   np.sin(yaw)

    R = np.stack([
        np.stack([cy*cp, cy*sp*sr - sy*cr, cy*sp*cr + sy*sr], axis=-1),
        np.stack([sy*cp, sy*sp*sr + cy*cr, sy*sp*cr - cy*sr], axis=-1),
        np.stack([-sp,   cp*sr,            cp*cr           ], axis=-1)
    ], axis=-2)

    return R


In [ ]:
# Define a function to convert the string representation of an array/list
def string_to_array(array_string):
    try:
        # Use ast.literal_eval to safely evaluate the string as a Python literal
        return ast.literal_eval(array_string)
    except (ValueError, SyntaxError):
        # Handle cases where the string might not be a valid list/array string
        return array_string # Or some default/error value

In [ ]:
def parse_exp_filename(filepath):
    """
    Parse filenames of the form:
        {approach}_model_{terrain}.csv

    Returns
    -------
    approach : str
    terrain  : str
    """
    fname = os.path.basename(filepath)
    stem, ext = os.path.splitext(fname)

    if ext.lower() != ".csv" or "_model_" not in stem:
        raise ValueError(
            f"Expected filename of the form '{{approach}}_model_{{terrain}}.csv', got: {fname}"
        )

    approach, terrain = stem.split("_model_", 1)
    return approach, terrain


def aggregate_terrain_data_by_approach(exp_dir,
                                       converters=None,
                                       recursive=False,
                                       sort_files=True,
                                       verbose=True):
    """
    Aggregate all experiment CSVs across terrains for each approach.

    The expected filename convention is:
        {approach}_model_{terrain}.csv

    Parameters
    ----------
    exp_dir : str
        Directory containing the experiment CSV files.
    converters : dict or None
        Optional converters passed to pd.read_csv.
    recursive : bool
        If True, search exp_dir recursively for CSV files.
    sort_files : bool
        If True, sort matched files for deterministic ordering.
    verbose : bool
        If True, print a brief summary.

    Returns
    -------
    approach_to_df : dict[str, pd.DataFrame]
        Dictionary mapping each approach name to a single concatenated dataframe
        containing all rows from all terrains evaluated for that approach.
    combined_df : pd.DataFrame
        One dataframe containing every row from every matched file, with the
        added metadata columns:
            - approach
            - terrain
            - source_file
    """
    if recursive:
        csv_files = []
        for root, _, files in os.walk(exp_dir):
            for f in files:
                if f.lower().endswith(".csv") and "_model_" in f:
                    csv_files.append(os.path.join(root, f))
    else:
        csv_files = [
            os.path.join(exp_dir, f)
            for f in os.listdir(exp_dir)
            if f.lower().endswith(".csv") and "_model_" in f
        ]

    if sort_files:
        csv_files = sorted(csv_files)

    if len(csv_files) == 0:
        raise FileNotFoundError(
            f"No CSV files matching '{{approach}}_model_{{terrain}}.csv' were found in: {exp_dir}"
        )

    frames = []
    for csv_path in csv_files:
        approach, terrain = parse_exp_filename(csv_path)
        df_i = pd.read_csv(csv_path, converters=converters)

        df_i["approach"] = approach
        df_i["terrain"] = terrain
        df_i["source_file"] = os.path.basename(csv_path)

        frames.append(df_i)

    combined_df = pd.concat(frames, axis=0, ignore_index=True)

    approach_to_df = {
        approach: df_group.reset_index(drop=True)
        for approach, df_group in combined_df.groupby("approach", sort=False)
    }

    if verbose:
        print(f"Matched {len(csv_files)} CSV files in: {exp_dir}")
        for approach, df_group in approach_to_df.items():
            terrains = sorted(df_group["terrain"].dropna().unique().tolist())
            print(
                f"  {approach}: {len(df_group)} rows aggregated across "
                f"{len(terrains)} terrain(s) -> {terrains}"
            )

    return approach_to_df, combined_df

In [ ]:
def compute_per_metric_per_timestep_errors(
    df,
    target_height=0.33,
    drop_failures=True
):
    """
    Compute per-timestep error metrics and return a long-form dataframe
    with one row per metric per timestep.

    Expected columns:
        - approach
        - terrain
        - failure
        - base_cmd       : [vx_cmd, vy_cmd, wz_cmd]
        - base_pose      : [x, y, z]
        - base_lin_vel   : [vx, vy, vz]
        - base_ang_vel   : [wx, wy, wz]
        - proj_grav      : [gx, gy, gz]

    Returns columns:
        - approach
        - terrain
        - metric
        - value

    Notes:
        - Scalars use MAE = abs(error)
        - Vector errors use L1 norm = sum(abs(...))
    """
    out_df = df.copy()

    if drop_failures and "failure" in out_df.columns:
        out_df = out_df.loc[out_df["failure"] == 0].copy()

    # Convert list-like columns to arrays
    base_cmd = np.asarray(out_df["base_cmd"].to_list(), dtype=float)         # [N, 3]
    base_pose = np.asarray(out_df["base_pose"].to_list(), dtype=float)       # [N, 3]
    base_lin_vel = np.asarray(out_df["base_lin_vel"].to_list(), dtype=float) # [N, 3]
    base_ang_vel = np.asarray(out_df["base_ang_vel"].to_list(), dtype=float) # [N, 3]
    proj_grav = np.asarray(out_df["proj_grav"].to_list(), dtype=float)       # [N, 3]

    # Per-timestep metrics
    lin_cmd_error_vec = base_cmd[:, 0:2] - base_lin_vel[:, 0:2]
    ang_cmd_error = base_cmd[:, 2] - base_ang_vel[:, 2]
    total_cmd_error_vec = np.concatenate(
        [lin_cmd_error_vec, ang_cmd_error[:, None]], axis=1
    )

    height_error = target_height - base_pose[:, 2]
    orientation_rp_vec = proj_grav[:, 0:2]
    ang_vel_rp_vec = base_ang_vel[:, 0:2]
    z_vel = base_lin_vel[:, 2]
    total_unwanted_vel_vec = np.concatenate(
        [z_vel[:, None], base_ang_vel[:, 0:2]], axis=1
    )

    metric_arrays = {
        "lin_cmd": np.linalg.norm(lin_cmd_error_vec, ord=1, axis=1),
        "ang_cmd": np.abs(ang_cmd_error),
        # "total_cmd_tracking_l1": np.linalg.norm(total_cmd_error_vec, ord=1, axis=1),
        "height": np.abs(height_error),
        "proj_grav": np.linalg.norm(orientation_rp_vec, ord=1, axis=1),
        "ang_vel": np.linalg.norm(ang_vel_rp_vec, ord=1, axis=1),
        "z_vel": np.abs(z_vel),
        # "total_unwanted_vel_l1": np.linalg.norm(total_unwanted_vel_vec, ord=1, axis=1),
    }

    rows = []
    approaches = out_df["approach"].values
    terrains = out_df["terrain"].values

    for metric_name, values in metric_arrays.items():
        rows.append(pd.DataFrame({
            "approach": approaches,
            "terrain": terrains,
            "metric": metric_name,
            "value": values,
        }))

    metrics_long_df = pd.concat(rows, ignore_index=True)
    return metrics_long_df

In [ ]:
# Example usage:
#
# exp_dir = "exp_data/kite_feasibility"
# csv_converters = {
#     'base_cmd': string_to_array,
#     'base_pose': string_to_array,
#     'base_rpy': string_to_array,
#     'q_actual': string_to_array,
#     'base_lin_vel': string_to_array,
#     'base_ang_vel': string_to_array,
#     'dof_vel': string_to_array,
#     'proj_grav': string_to_array,
#     'feet_pos': string_to_array,
#     'tau_act': string_to_array,
#     'grf': string_to_array,
#     'q_des': string_to_array,
#     'tau_ff': string_to_array,
#     'tau_pd': string_to_array,
#     'failure': string_to_array,
# }
#
# approach_dfs, all_exp_df = aggregate_terrain_data_by_approach(
#     exp_dir,
#     converters=csv_converters,
# )
#
# # Access the aggregated dataframe for a single approach:
# # baseline_df = approach_dfs["baseline"]

In [ ]:
exp_dir = "exp_data/kite_feasibility"
csv_converters = {
    'base_cmd': string_to_array,
    'base_pose': string_to_array,
    'base_rpy': string_to_array,
    'q_actual': string_to_array,
    'base_lin_vel': string_to_array,
    'base_ang_vel': string_to_array,
    'dof_vel': string_to_array,
    'proj_grav': string_to_array,
    'feet_pos': string_to_array,
    'tau_act': string_to_array,
    'grf': string_to_array,
    'q_des': string_to_array,
    'tau_ff': string_to_array,
    'tau_pd': string_to_array,
    'failure': string_to_array,
}

approach_dfs, all_exp_df = aggregate_terrain_data_by_approach(
    exp_dir,
    converters=csv_converters,
)

In [ ]:
viz_dataframe = compute_per_metric_per_timestep_errors(all_exp_df) 

In [ ]:
ax = sns.boxplot(viz_dataframe, x="metric", y="value", hue="approach", showfliers=False)

In [ ]:
exp_filepath = "exp_data/kite_feasibility/baseline_model_plane.csv"
# exp_filepath = "exp_data/kite_feasibility/kite_model_plane.csv"


df = pd.read_csv(exp_filepath, converters={'base_cmd': string_to_array,
                                           'base_pose': string_to_array,
                                           'base_rpy': string_to_array,
                                           'q_actual': string_to_array,
                                           'base_lin_vel': string_to_array,
                                           'base_ang_vel': string_to_array,
                                           'dof_vel': string_to_array,
                                           'proj_grav': string_to_array,
                                           'feet_pos': string_to_array,
                                           'tau_act': string_to_array,
                                           'grf': string_to_array,
                                           'q_des': string_to_array,
                                           'tau_ff': string_to_array,
                                           'tau_pd': string_to_array,
                                           'failure': string_to_array})

print(len(df))

In [ ]:
# base_cmd	base_pose	base_rpy	q_actual	base_lin_vel	base_ang_vel	dof_vel	proj_grav	feet_pos


# Joint tracking / violation metrics
q_observations = np.array(df.loc[df["failure"] == 0, "q_actual"].to_list())

# cmd tracking metrics
vel_cmds = np.array(df.loc[df["failure"] == 0, "base_cmd"].to_list())  # v_x, v_y, w_yaw
lin_vel  = np.array(df.loc[df["failure"] == 0, "base_lin_vel"].to_list())
ang_vel  = np.array(df.loc[df["failure"] == 0, "base_ang_vel"].to_list())   # stability via roll/pitch angular velo.

# Oreintation
proj_grav = np.array(df.loc[df["failure"] == 0, "proj_grav"].to_list())

# Height tracking metric
base_pose = np.array(df.loc[df["failure"] == 0, "base_pose"].to_list())
base_rpy = np.array(df.loc[df["failure"] == 0, "base_rpy"].to_list())

R_w_b = rpy_to_rotmat(base_rpy)

joint_limits = np.array([[-1.047, -0.633, -2.721, -1.047, -0.633, -2.721, -1.047, -0.633, -2.721, -1.047, -0.633, -2.721],
                         [1.047, 2.966, -0.837, 1.047, 2.966, -0.837, 1.047, 2.966, -0.837, 1.047, 2.966, -0.837]])

joint_torque_limits = np.array([23.7, 23.7, 35.55, 23.7, 23.7, 35.55, 23.7, 23.7, 35.55, 23.7, 23.7, 35.55])

In [ ]:
print("Number of failures: ", df["failure"].sum())

In [ ]:
lin_cmd_errors = vel_cmds[:,0:2] - lin_vel[:,0:2]
ang_cmd_errors = vel_cmds[:,2] - ang_vel[:,2]

cmd_errs = np.concatenate((lin_cmd_errors, ang_cmd_errors[:,None]), axis=1)

print(cmd_errs.shape)

print("Linear CMD Tracking  RMSE: ", np.round(np.sqrt(np.mean(np.square(lin_cmd_errors))),4))
print("Linear CMD Tracking STDDV: ", np.round(np.sqrt(np.std(np.square(lin_cmd_errors))),4))
print("Linear CMD Tracking  MAE: ", np.round(np.mean(np.abs(lin_cmd_errors)),4))
print("Linear CMD Tracking  Median: ", np.round(np.median(np.abs(lin_cmd_errors)),4))


print("Angular CMD Tracking  RMSE: ", np.round(np.sqrt(np.mean(np.square(ang_cmd_errors))),4))
print("Angular CMD Tracking STDDV: ", np.round(np.sqrt(np.std(np.square(ang_cmd_errors))),4))
print("Angular CMD Tracking  MAE: ", np.round(np.mean(np.abs(ang_cmd_errors)),4))
print("Angular CMD Tracking  Median: ", np.round(np.median(np.abs(ang_cmd_errors)),4))


print("Total CMD Tracking  RMSE: ", np.round(np.sqrt(np.mean(np.square(cmd_errs))),4))
print("Total CMD Tracking  STDDV: ", np.round(np.sqrt(np.std(np.square(cmd_errs))),4))

In [ ]:
print("Max Linear Velcoity",  np.max(lin_vel[:,0:2], axis=0))
print("Max Angular Velocity",  np.max(ang_vel[:,2], axis=0))

In [ ]:
height_errors = 0.33 - base_pose[:,2]

print("Height CMD Tracking  RMSE: ", np.round(np.sqrt(np.mean(np.square(height_errors))),4))
print("Height CMD Tracking  MAE: ", np.round(np.mean(np.abs(height_errors)),4))

In [ ]:
orientation_norm = np.linalg.norm(proj_grav[:,0:2], axis=1, ord=1)

print("Projected Grav. R/P Norm: ", np.round(np.mean(orientation_norm),4))

In [ ]:
velo_z = lin_vel[:,2]
ang_velo_norm = np.linalg.norm(ang_vel[:,0:2], axis=1, ord=1)

print("Angular Velo. R/P Norm MEAN: ", np.round(np.mean(ang_velo_norm),4))
print("Angular Velo. R/P Norm STD: ", np.round(np.std(ang_velo_norm),4))

print("Z Velo. MEAN: ", np.round(np.sqrt(np.mean(np.square(velo_z))),4))
print("Z Velo. STD: ",  np.round(np.sqrt(np.std(np.square(velo_z))),4))
print("Z Velo. MAE: ",  np.round(np.mean(np.abs(velo_z)),4))

total_unwatnted_velo = np.concatenate((velo_z[:,None], ang_vel[:,0:2]), axis=1)

print("Total Velo. R/P Norm MEAN: ", np.round(np.mean(np.linalg.norm(total_unwatnted_velo, axis=1)),4))
print("Angular Velo. R/P Norm STD: ", np.round(np.std(np.linalg.norm(total_unwatnted_velo, axis=1)),4))